# 04 · Evaluación

Reglas de decisión (argmax, umbral binario, mínimo costo esperado), costo total en test y bootstrap pareado al 95 %.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append('..')
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
import utils

MODELS = ['lgbm', 'hazard', 'ordinal']
FORMS  = ['convencional', 'landmarking']

In [ ]:
class PriorShift:
    """Reescala cada clase por π_objetivo / π_origen y renormaliza."""
    def fit(self, P, y, w=None):
        pi_target = np.bincount(np.asarray(y, int), minlength=5) + 1.0
        self.r_ = (pi_target / pi_target.sum()) / P.mean(0)
        return self
    def transform(self, P):
        Q = P * self.r_
        return Q / Q.sum(1, keepdims=True)

methods = {'sin calibrar': None,
           'temperatura': utils.TemperatureScaler,
           'isotónica':   utils.IsotonicOVR,
           'prior':       PriorShift}

def crossfit_eval(Pv, yv, method, k=5, seed=0):
    """Calibración y umbral ajustados en k-1 grupos de validation, evaluados en el restante."""
    yhat = {r: np.zeros(len(yv), int) for r in ['argmax', 'umbral', 'min_costo']}
    Q = np.zeros_like(Pv)
    for tr, te in StratifiedKFold(k, shuffle=True, random_state=seed).split(Pv, yv):
        C = None if method is None else method().fit(Pv[tr], yv[tr])
        Qtr = Pv[tr] if C is None else C.transform(Pv[tr])
        Qte = Pv[te] if C is None else C.transform(Pv[te])
        theta, _ = utils.best_threshold(Qtr, yv[tr])
        yhat['argmax'][te]    = utils.decide_argmax(Qte)
        yhat['umbral'][te]    = utils.decide_threshold(Qte, theta)
        yhat['min_costo'][te] = utils.decide_min_cost(Qte)
        Q[te] = Qte
    return Q, yhat

In [ ]:
rows, preds_cf = [], {}
for model in MODELS:
    for form in FORMS:
        val = utils.load_predictions(model, form, 'validation')
        Pv, yv = val[utils.PRED_COLS].to_numpy(), val['y'].to_numpy()
        for mname, M in methods.items():
            Q, yhat = crossfit_eval(Pv, yv, M)
            preds_cf[(model, form, mname)] = yhat
            for rule, yh in yhat.items():
                rows.append({'modelo': model, 'formulación': form, 'calibración': mname,
                             'regla': rule, 'Brier': round(utils.brier(yv, Q), 4),
                             'costo_total': utils.total_cost(yv, yh),
                             '% a revisión': round(100 * (yh > 0).mean(), 1)})

tab_cf = pd.DataFrame(rows)
tab_cf.to_csv(utils.RESULTS / 'reglas_validacion_cruzada_lgbm_hazard_v2.csv', index=False)

# Calibración elegida = menor Brier (regla del diseño)
brier_tab = tab_cf.groupby(['modelo', 'formulación', 'calibración'])['Brier'].first()
best_cal = brier_tab.groupby(['modelo', 'formulación']).idxmin().map(lambda t: t[2]).to_dict()
print('Calibración elegida:', best_cal, '\n')

sel = tab_cf[[best_cal[(m, f)] == c for m, f, c in
              zip(tab_cf['modelo'], tab_cf['formulación'], tab_cf['calibración'])]]
sel.pivot_table(index=['modelo', 'formulación'], columns='regla',
                values=['costo_total', '% a revisión'])

In [ ]:
def pick(model, form, rule):
    return preds_cf[(model, form, best_cal[(model, form)])][rule]

comparaciones = [
    ('H2 lgbm conv: min_costo vs argmax',   ('lgbm', 'convencional', 'min_costo'),   ('lgbm', 'convencional', 'argmax')),
    ('H2 hazard conv: min_costo vs argmax', ('hazard', 'convencional', 'min_costo'), ('hazard', 'convencional', 'argmax')),
    ('Umbral vs min_costo (lgbm conv)',     ('lgbm', 'convencional', 'umbral'),      ('lgbm', 'convencional', 'min_costo')),
    ('Umbral vs min_costo (hazard conv)',   ('hazard', 'convencional', 'umbral'),    ('hazard', 'convencional', 'min_costo')),
    ('H1 lgbm: landmarking vs conv',        ('lgbm', 'landmarking', 'umbral'),       ('lgbm', 'convencional', 'umbral')),
    ('H1 hazard: landmarking vs conv',      ('hazard', 'landmarking', 'umbral'),     ('hazard', 'convencional', 'umbral')),
    ('Modelo: hazard vs lgbm (conv)',       ('hazard', 'convencional', 'umbral'),    ('lgbm', 'convencional', 'umbral')),
    ('Mejor lgbm vs siempre 4',             ('lgbm', 'convencional', 'umbral'),      None),
    ('Mejor hazard vs siempre 4',           ('hazard', 'convencional', 'umbral'),    None),
    ('H2 ordinal conv: min_costo vs argmax', ('ordinal', 'convencional', 'min_costo'), ('ordinal', 'convencional', 'argmax')),
    ('H1 ordinal: landmarking vs conv',      ('ordinal', 'landmarking', 'umbral'),     ('ordinal', 'convencional', 'umbral')),
    ('Modelo: ordinal vs lgbm (conv)',       ('ordinal', 'convencional', 'umbral'),    ('lgbm', 'convencional', 'umbral')),
    ('Mejor ordinal vs siempre 4',           ('ordinal', 'convencional', 'umbral'),    None),
]

rows = []
for desc, a, b in comparaciones:
    ya = pick(*a)
    yb = np.full(len(yv), 4) if b is None else pick(*b)
    rows.append({'comparación': desc, **utils.paired_bootstrap(yv, ya, yb)})

boot = pd.DataFrame(rows)
boot.to_csv(utils.RESULTS / 'bootstrap_validation_lgbm_hazard.csv', index=False)
boot

- La regla de decisión es lo que más importa. Pasar de «la clase más probable» a una regla sensible al costo ahorra unos 22 000. Es entre 7 y 30 veces más que cualquier otra decisión del diseño. H2 se cumple con claridad en los dos modelos.
- El umbral y el mínimo costo empatan una vez que cada modelo usa su mejor calibración. La diferencia que vimos antes en gradient boosting venía de haber usado una calibración no óptima.
- El modelo casi no importa. Una regresión logística de riesgo discreto rinde igual que gradient boosting. Si el Transformer también empata, H3 queda confirmada.
- La formulación tampoco importa en este benchmark (H1 no se cumple). Ya tenemos la explicación: las etiquetas de evaluación usan la convención antigua.

Excluir los censurados, que es lo que parece «más correcto» en un clasificador, puede ser peor que etiquetarlos como sanos, y solo un modelo de supervivencia los aprovecha bien.